In [ ]:
#| default_exp features.tfbs_genomewide


In [ ]:
#| export
from __future__ import annotations
import pandas as pd
import structlog
from kreview.eval_engine import FeatureEvaluator

log = structlog.get_logger()


In [ ]:
#| export
class TFBSEvaluator(FeatureEvaluator):
    """Extracts TFBS footprint metrics."""
    
    name = "TfbsGenomewide"
    source_file = ".TFBS.parquet"
    tier = 2
    category = "epigenetics_and_geometry"

    def extract(self, df: pd.DataFrame) -> dict[str, float]:
        extracted = {}
        try:
            if df.empty:
                return extracted
            cols = set(df.columns)

            if "label" in cols:
                metrics = [c for c in ["count", "mean_size", "entropy", "z_score"] if c in cols]
                for row in df.to_dict('records'):
                    lbl = str(row["label"]).replace(" ", "_").replace("-","_")
                    for m in metrics:
                        if pd.notna(row[m]):
                            extracted[f"{lbl}_{m}"] = float(row[m])
    
            return extracted
        except Exception as e:
            log.exception("extraction_failed", evaluator=self.name, error=str(e))
            return {}


In [ ]:
#| test
evaluator = TFBSEvaluator()
df = pd.DataFrame([{"label":"CTCF", "entropy": 5.0, "z_score": -2.0}])
res = evaluator.extract(df)
assert res["CTCF_entropy"] == 5.0
